# Survivor Data Set: An Exploration
## Project Overview

We seek to understand the underlying themes in the reality TV gameshow "Survivor". The show is currently airing it's 50th season, but the gameplay we see in the later seasons is very different from the early seasons of the show. Survivor is a gameshow about physical challenges and mental strength, but fundamentally it is a show about relationships with others. In a game where the goal is to "Outwit, Outplay, Outlast", what does this actually translate to? In order to win you need the votes of the jury, which is made up of the players you had a hand in voting off of the tribe.

The thing that makes Survivor so interesting is how different every season is, and how the game is always evolving. In early seasons, loyalty and morals were valued, and the gameplay was relatively simple. As time went on, players got more conniving, and blindsides and backstabbing became the norm. But what is most interesting is that as the gameplay changed, so did the mentality of the players. Playing fair isn't enough to win anymore. 

Through this data we hope to find answers to some of these questions:
- What does it take Outwit, Outplay, and Outlast?
- What decisions to winners make? 
- What decisions do losers make?
- How have the decisions made by winners and losers changed as the game has evolved?
- Are there trends that predict performance?
- Are there qualities that predict performance?
- Have there been changes in these trends/qualities over time?

## Data Description and Source
The original survivoR dataset is in R, [original data set](https://github.com/doehm/survivoR/tree/master/data). We will be using a modified version of this dataset that has been converted to CSV files found here: [Survivor Data Set](https://github.com/rfordatascience/tidytuesday/tree/master/data/2021/2021-06-01)

The original dataset has 23 R data files, while the dataset that's been converted to CSV only has 5 of these. 

The 5 csv files are:
- Summary
- Challanges
- Castaways
- Viewers
- Jury Votes


# Initial CSV Data
The sections below walk through each CSV in the same order: **Initial CSV exploration**, **Cleaning needs**, **Methods**, and **Results**.


## Setup: imports and loading

Import **pandas**, **matplotlib** / **seaborn** for plots, **IPython.display** for styled tables, then load all tables from the TidyTuesday CSV mirror.


In [ ]:
# Imports for the project
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

In [ ]:
# Each table is available as a CSV from the TidyTuesday GitHub mirror
base_url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-06-01/"

# Load the main tables
summary_df     = pd.read_csv(base_url + "summary.csv")
challenges_df  = pd.read_csv(base_url + "challenges.csv")
castaways_df   = pd.read_csv(base_url + "castaways.csv")
viewers_df     = pd.read_csv(base_url + "viewers.csv")
jury_votes_df  = pd.read_csv(base_url + "jury_votes.csv")

## Table: Summary


### Initial CSV exploration

#### Relevance:
- Primary data set for information on seasons, winners, and dates.
- Useful for determining winners and trends over time.

#### Size:
- 40 rows
- 19 columns
- 6.1 KB

#### Column descriptions (English):
- **Season Name**: The name of the season
- **Season**: The season number
- **Location**: The geographical location of the show
- **Country**: The country where the season takes place
- **Tribe Setup**: How players are divided into tribes (teams)
- **Full Name**: The name of the player
- **Winner**: The winner of the season
- **Runner Ups**: Second place
- **Final Vote**: The vote split for the winner
- **Time Slot**: Day and time of week when episodes aired
- **Premiered**: Date when the season premiered
- **Ended**: Date when the season ended
- **Filming Strated**: Date when filming started (note: this spelling matches the CSV column label)
- **Filming Ended**: Date when filming ended
- **Viewers Finale**: Number of viewers for the final episode
- **Viewers Reunion**: Number of viewers for the reunion (contestant retrospective)
- **Viewers Mean**: Average viewership for episodes
- **Rank**: Viewer final ranking of the contestant

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
summary_df.info()
summary_df.head()

### Cleaning

All steps use `clean_summary_df`, created with `summary_df.copy()` in the first cleaning code cell, so `summary_df` stays the untouched load. From raw inspection, we fix:

- `premiered`, `ended`, `filming_started`, and `filming_ended`: strings → datetimes
- `timeslot`: split into `weekday` and `air_time`
- `viewers_mean` and `rank`: median fill for `viewers_mean` and missing `rank` kept with `has_rank` for training filters


### Methods

The following code cells implement the cleaning steps on `clean_summary_df`.


In [ ]:
# Working copy for cleaning; summary_df remains the raw CSV load
clean_summary_df = summary_df.copy()

In [ ]:
# View the timeslot column entries
clean_summary_df['timeslot'].value_counts()

In [ ]:
# Extract weekday and time text from timeslot
parts = clean_summary_df["timeslot"].str.extract(r"(\w+)\s+(.+)")
clean_summary_df["weekday"] = parts[0]
clean_summary_df["air_time_raw"] = parts[1]

# Parse time text into a time value
clean_summary_df["air_time"] = pd.to_datetime(
    clean_summary_df["air_time_raw"].str.strip().str.lower(),
    format="%I:%M %p",
    errors="coerce"
).dt.time

# Drop the timeslot column and airtime_raw column
clean_summary_df = clean_summary_df.drop(columns=["timeslot", "air_time_raw"])

clean_summary_df.info()

In [ ]:
# Convert the premiered, ended, filming_started, and filming_ended columns to datetime
clean_summary_df['premiered'] = pd.to_datetime(clean_summary_df['premiered'])
clean_summary_df['ended'] = pd.to_datetime(clean_summary_df['ended'])
clean_summary_df['filming_started'] = pd.to_datetime(clean_summary_df['filming_started'])
clean_summary_df['filming_ended'] = pd.to_datetime(clean_summary_df['filming_ended'])

# Show the updated dataframe
clean_summary_df.info()

In [ ]:
# View the null values in viewers_mean and rank
clean_summary_df[clean_summary_df['viewers_mean'].isna() | clean_summary_df['rank'].isna()]

`rank` and `viewers_mean` are missing in the same two rows. We keep every season, so `viewers_mean` is filled with the median of non-missing values, `rank` stays NaN where unknown, and `has_rank` is True only when `rank` is present so rows without a label can be excluded when training a model that predicts `rank`.

In [ ]:
# Median impute viewers_mean; keep missing rank and flag rows usable as labeled training data
viewers_median = clean_summary_df["viewers_mean"].median()
clean_summary_df["viewers_mean"] = clean_summary_df["viewers_mean"].fillna(
    viewers_median
)
clean_summary_df["has_rank"] = clean_summary_df["rank"].notna()

clean_summary_df.info()

### Results

**Summary:** Summary Data Frame

`clean_summary_df` keeps all 40 seasons with consistent dtypes and `weekday` / `air_time` from `timeslot`. After cleaning, `viewers_mean` has no nulls, from median imputation; `rank` is still null for the two incomplete seasons, so `has_rank` marks rows to include when `rank` is a target.


## Table: Challenges


### Initial CSV exploration

#### Relevance:
- Examines outcomes when contestants and tribes compete; these outcomes influence decisions when tribes vote contestants out.

#### Size:
- 5023 rows
- 8 columns
- 314.1 KB

#### Column descriptions (English):
- **season_name**: The season's name
- **season**: The season number
- **episode**: The episode number within the season
- **title**: The title of the episode
- **day**: The running day count since the game began
- **challenge_type**: Reward (a prize) or immunity (protection from being voted out)
- **winners**: The name of the contestant who won the challenge
- **winning_tribe**: The name of the tribe that won the challenge

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
print(challenges_df.info())
print(challenges_df.head())

# Challenge nulls before cleaning
challenge_nulls_before = challenges_df[["winners", "winning_tribe"]].isna().sum()
print("Nulls before cleaning:")
print(challenge_nulls_before)

### Cleaning Needs / Methods

The following code cell houses all the challenge-row cleaning rules for `challenges_df`.


In [ ]:
# Clean Version
challenges_cleaned_df = challenges_df.copy()

# Dropping rows that are placeholder / non-challenge rows
drop_mask = (
    # Survivor: Island of the Idols, episode 12
    ((challenges_cleaned_df["season"] == 39) &
     (challenges_cleaned_df["episode"] == 12) &
     (challenges_cleaned_df["day"] == 36) &
     (challenges_cleaned_df["challenge_type"] == "immunity") &
     (challenges_cleaned_df["winners"].isna())) |

    # Survivor: David vs. Goliath, episode 4
    ((challenges_cleaned_df["season"] == 37) &
     (challenges_cleaned_df["episode"] == 4) &
     (challenges_cleaned_df["day"] == 10) &
     (challenges_cleaned_df["challenge_type"] == "immunity") &
     (challenges_cleaned_df["winners"].isna()))
)

challenges_cleaned_df = challenges_cleaned_df.loc[~drop_mask].copy()

# Filling in rows where there truly was no immunity winner
no_winner_cases = [
    (32, 13),  # Kaoh Rong (Joe evacuated)
    (24, 6),   # One World (Colton evacuated)
    (21, 12),  # Nicaragua (NaOnka and Kelly quit)
    (19, 6),   # Samoa (Russell Swan evacuated)
    (12, 11),  # Panama (Bruce evacuated)
    (8, 3),    # All-Stars (Jenna quit)
    (8, 6),    # All-Stars (Sue quit)
    (2, 6)     # Australian Outback (Michael evacuated)
]

for season_num, episode_num in no_winner_cases:
    mask = (
        (challenges_cleaned_df["season"] == season_num) &
        (challenges_cleaned_df["episode"] == episode_num) &
        (challenges_cleaned_df["challenge_type"] == "immunity") &
        (challenges_cleaned_df["winners"].isna())
    )

    challenges_cleaned_df.loc[mask, "winners"] = "No challenge winner"
    challenges_cleaned_df.loc[mask, "winning_tribe"] = "Not applicable"

# If a winner exists but winning_tribe is missing, then tribe winner does not apply
tribe_not_applicable_mask = (
    challenges_cleaned_df["winners"].notna() &
    challenges_cleaned_df["winning_tribe"].isna()
)

challenges_cleaned_df.loc[tribe_not_applicable_mask, "winning_tribe"] = "Not applicable"

# Fixing Survivor: Blood vs. Water, episode 1
# Galang won the combined immunity/reward challenge, so restore the missing immunity winners
bvw_bad_immunity_rows = (
    (challenges_cleaned_df["season"] == 27) &
    (challenges_cleaned_df["episode"] == 1) &
    (challenges_cleaned_df["day"] == 1) &
    (challenges_cleaned_df["challenge_type"] == "immunity") &
    (challenges_cleaned_df["winners"].isna())
)

challenges_cleaned_df = challenges_cleaned_df.loc[~bvw_bad_immunity_rows].copy()

galang_members = [
    "Aras",
    "Colton",
    "Gervase",
    "Kat",
    "Laura B.",
    "Laura M.",
    "Monica",
    "Tina",
    "Tyson"
]

bvw_immunity_rows = pd.DataFrame(
    [
        {
            "season_name": "Survivor: Blood vs. Water",
            "season": 27,
            "episode": 1,
            "title": "Blood Is Thicker Than Anything",
            "day": 1,
            "challenge_type": "immunity",
            "winners": member,
            "winning_tribe": "Galang"
        }
        for member in galang_members
    ]
)

challenges_cleaned_df = pd.concat(
    [challenges_cleaned_df, bvw_immunity_rows],
    ignore_index=True
)

# Drop the last 3 blank reward placeholder rows
final_placeholder_rows = (
    ((challenges_cleaned_df["season"] == 22) &
     (challenges_cleaned_df["episode"] == 14) &
     (challenges_cleaned_df["day"] == 38) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna())) |

    ((challenges_cleaned_df["season"] == 16) &
     (challenges_cleaned_df["episode"] == 1) &
     (challenges_cleaned_df["day"] == 3) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna())) |

    ((challenges_cleaned_df["season"] == 13) &
     (challenges_cleaned_df["episode"] == 6) &
     (challenges_cleaned_df["day"] == 15) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna()))
)

challenges_cleaned_df = challenges_cleaned_df.loc[~final_placeholder_rows].copy()

# Sort the cleaned df so the column names make sense
challenges_cleaned_df = challenges_cleaned_df.sort_values(
    by=["season", "episode", "day", "challenge_type", "winners"]
).reset_index(drop=True)

# Challenge nulls after cleaning
challenge_nulls_after = challenges_cleaned_df[["winners", "winning_tribe"]].isna().sum()
print("\nChallenge nulls after cleaning:")
print(challenge_nulls_after)

print(challenges_cleaned_df.info())
print(challenges_cleaned_df.head())

### Results

See the printed null counts and `challenges_cleaned_df.info()` / `head()` output above for this run.


## Table: Castaways


### Initial CSV exploration

#### Relevance:
- Personal data on each contestant and their performance.
- Useful for analyzing how traits relate to outcomes.

#### Size:
- 744 rows
- 18 columns
- 104.8 KB

#### Column descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **full_name**: The contestant's full name
- **castaway**: The castaway's (contestant's) first name
- **age**: The contestant's age
- **city**: The city the contestant is from
- **state**: The state the contestant is from
- **personality_type**: Their personality description
- **day**: The day of the season (running count)
- **order**: Finish order for the season (larger values mean the contestant lasted longer)
- **result**: When they were voted out (string description of the order column)
- **jury_status**: If and when the player made the jury
- **original_tribe**: The tribe they started on
- **swapped_tribe**: The tribe they swapped to
- **swapped_tribe2**: The tribe they swapped to a second time
- **merged_tribe**: The merged tribe name
- **total_votes_received**: Number of votes cast against the contestant
- **immunity_idols_won**: Number of immunity idols won by the contestant

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
castaways_df.info()
castaways_df.head()

#### Raw inspection (continued)

The following information is used to determine the basic information about the dataset that is needed, and explained at the top of this section.


In [ ]:
# Shape and column names
print("SHAPE")
print(castaways_df.shape)

print("\nCOLUMNS")
print(castaways_df.columns.tolist())

# Data types
print("\nDATA TYPES")
print(castaways_df.dtypes)

# Missing values
print("\nMISSING VALUES")
print(castaways_df.isnull().sum())


### Cleaning needs

#### Data cleaning: Fixing null values

Based on the above cell we find the following columns have null values:
| Column Name | Null Count |
|--------|-----------|
| personality_type | 3 |
| jury_status | 405 |
| original_tribe | 2 |
| swapped_tribe | 284 |
| swapped_tribe2 | 683 |
| merged_tribe | 300 |


### Methods

The following code cells implement the castaways cleaning steps on `clean_castaways_df`.


In [ ]:
#first we create a copy of the dataframe for editing
clean_castaways_df = castaways_df.copy()

#### personality_type
First we will fix the missing personality types. Since we are only missing three, we could just drop these rows, but we may want other information on these players, so we will fill them with 'UNKNOWN' instead.

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['personality_type'].info())

In [ ]:
print(clean_castaways_df['personality_type'].unique())
clean_castaways_df['personality_type'] = clean_castaways_df['personality_type'].fillna('UNKNOWN')

In [ ]:
print("After: \n")
print(clean_castaways_df['personality_type'].info())

#### jury_status

In [ ]:
# See all unique values in the column
print(f"Before: \n")
print(clean_castaways_df['jury_status'].info())

In [ ]:
print(castaways_df['jury_status'].unique())

After viewing this we can understand that this column tells us what member of the jury a castaway is. The players eliminated earlier in the season don't make it on the jury, which explains why there are so many null values. After some consideration we will replace all null values with "non jury member" to follow the naming convention. 

In [ ]:
clean_castaways_df['jury_status'] = clean_castaways_df['jury_status'].fillna('non jury member')

In [ ]:
print("After: \n")
print(clean_castaways_df['jury_status'].info())

#### original_tribe
Since there are only two null values in this column, we will investigate which two castaways have the null values to determine if we should drop the rows or replace the null with a value.

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['original_tribe'].info())

In [ ]:
#output the two null rows
print(castaways_df[castaways_df['original_tribe'].isnull()])

After some googling I found this explanation from a Reddit post: "For those of you who don’t know, in the very first episode of Palau (Season 10), Jonathan Libby and Wanda Shirk were eliminated before tribes were even officially formed." Based on this we can conclude that we can drop these rows from the dataframe since they are inconsequential to understanding themes in the show. 

In [ ]:
clean_castaways_df = clean_castaways_df.dropna(subset=['original_tribe'])


In [ ]:
print("After: \n")
clean_castaways_df['original_tribe'].info()

#### swapped_tribe, swapped_tribe2
We will handle swapped_tribe and swapped_tribe2 together since they contain the same type of information, but some castaways swap tribes once, twice, or never.

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['swapped_tribe'].info())
print(clean_castaways_df['swapped_tribe2'].info())

In [ ]:
print(clean_castaways_df['swapped_tribe'].unique())
print(clean_castaways_df['swapped_tribe2'].unique())

From this output and some general knowledge, we can understand that these columns tell us what tribe a castaway switched to. However, not all players switch tribes, and especially most players don't switch tribes twice, but they are still important to the story the data is telling us. We will fill these with "Not Applicable".

In [ ]:
swap_columns = ['swapped_tribe', 'swapped_tribe2']
clean_castaways_df[swap_columns] = clean_castaways_df[swap_columns].fillna('Not Applicable')

In [ ]:
print("After: \n")
clean_castaways_df[swap_columns].info()

#### merged_tribe

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['merged_tribe'].info())

In [ ]:
print(clean_castaways_df['merged_tribe'].unique())
print(f"Number of merged tribe names: {clean_castaways_df['merged_tribe'].nunique()}")

After viewing this list of the merged tribe names we now understand that this column represents the name of the merged tribe a castaway was in. Part way through the season all of the tribes get merged into one tribe, and they create a new name for the tribe. Since around half of the players each season get eliminated before the merge, said players have null values. We will fill this with 'Not Applicable' since we still want the data on the players who don't make it to the merge.

In [ ]:
clean_castaways_df['merged_tribe'] = clean_castaways_df['merged_tribe'].fillna('Not Applicable')

In [ ]:
print("After: \n")
clean_castaways_df['merged_tribe'].info()

### Results


#### The data for the Castaways dataframe is now cleaned.

In [ ]:
clean_castaways_df.info()

## Table: Viewers


### Initial CSV exploration

#### Relevance:
- Useful for gauging interest in Survivor.
- May reveal relationships between public interest and contestant performance.

#### Size:
- 596 rows
- 9 columns
- 42 KB

#### Column descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **episode_number_overall**: The episode number across all seasons
- **episode**: The episode number within this season
- **title**: The title of the episode
- **episode_date**: The date the episode aired
- **viewers**: The number of viewers, in millions
- **rating_18_49**: Percentage of TV households in the 18-49 demographic that watched Survivor
- **share_18_49**: Among 18-49 viewers watching TV during the time slot, the percentage who watched Survivor

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
viewers_df.info()
viewers_df.head()

### Cleaning needs


First we make a copy of the dataframe to clean.

#### episode_number_overall
The overall episode numbers are stored as floats, but there is never an episode with a fraction, so we will convert this coumn to ints. First we need to investigate the single null value.

We can see that episode 2 of this season is 501, and after a google check as well, season 34 episode 1 of survivor is indeed the 500th episode, so we will manually change it, and then convert all of the values in this column to integers since an episode number won't be anything but a whole number. 

#### episode_date
episode_date is currently stored as an object, but we can convert it to datetime to be more accurate to what is is representing.

#### viewers, rating, share

Based on these observations it seems like there is just missing data for viewers, ratings, and the shares. We will take the average of each of these categories to fill in the data. This may skew things in a graph or other ways which we saw in class, but it is the best option to keep all of the episode details. 

The rating and share columns are confusing and lengthy with the addition of the 18_49 at the end, especially because there is no other age range as a different dataset, so there's no need for that distinction. We will change the column names to rating and share for clarity and ease.

### Methods


In [ ]:
clean_viewers_df = viewers_df.copy()

In [ ]:
null_idx = clean_viewers_df[clean_viewers_df['episode_number_overall'].isnull()].index[0]
print(clean_viewers_df.loc[null_idx - 1 : null_idx + 1])

In [ ]:
clean_viewers_df.loc[84, 'episode_number_overall'] = 500
clean_viewers_df['episode_number_overall'] = clean_viewers_df['episode_number_overall'].astype('int64')

In [ ]:
clean_viewers_df['episode_date'] = pd.to_datetime(clean_viewers_df['episode_date'])

In [ ]:
print("Before: \n")
print(clean_viewers_df['viewers'].info())
print("\n")
print(clean_viewers_df['rating_18_49'].info())
print("\n")
print(clean_viewers_df['share_18_49'].info())


In [ ]:

print(viewers_df[viewers_df['viewers'].isnull()].head())

In [ ]:
median_viewers = clean_viewers_df['viewers'].median()
median_rating = clean_viewers_df['rating_18_49'].median()
median_share = clean_viewers_df['share_18_49'].median()

clean_viewers_df['viewers'] = clean_viewers_df['viewers'].fillna(median_viewers)
clean_viewers_df['rating_18_49'] = clean_viewers_df['rating_18_49'].fillna(median_rating)
clean_viewers_df['share_18_49'] = clean_viewers_df['share_18_49'].fillna(median_share)

In [ ]:
clean_viewers_df = clean_viewers_df.rename(columns={
    'rating_18_49': 'rating',
    'share_18_49': 'share'
})

### Results


In [ ]:
print("After: \n")
print(clean_viewers_df['episode_number_overall'].info())

In [ ]:
print("After: \n")
print(clean_viewers_df['episode_date'].info())

In [ ]:
print("After: \n")
print(clean_viewers_df['viewers'].info())
print("\n")
print(clean_viewers_df['rating'].info())
print("\n")
print(clean_viewers_df['share'].info())

#### Viewers dataframe is now cleaned:

In [ ]:
print(clean_viewers_df.info())

## Table: Jury votes


### Initial CSV exploration

#### Relevance:
- Captures how jury members voted at Final Tribal Council.
- May reveal how personalities affect outcomes in the end.

#### Size:
- 909 rows
- 5 columns
- 35.6 KB

#### Column descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **castaway**: The juror casting votes
- **finalist**: A finalist for the season who can receive jury votes
- **vote**: Whether the juror voted for this finalist (1 = yes, 0 = no)

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
jury_votes_df.info()
jury_votes_df.head()

### Cleaning needs

No cleaning steps for this dataframe yet. There are no nulls, appropriate column names, and column types.


## Sources Used

- **Tool:** Cursor
- **How used:** Structure, cell order, format, PEP 8, grammatical errors, syntax errors, semantic errors
- **Scope:** Refinement, Debugging, Review

# Statistics

Now we focus on some statistics to determin what it takes to outwit, outplay, and outlast!

The following statistics will calculated and interpreted:
- Mean
- Median
- Standard Deviation
- Correlations


# Defining Winners & Losers

The following data are derived for a clean analysis phase

Identity
- `season` (int)
- `castaway` (short name)
- `full_name` (labels and tie-breaks)

Personality
- `personality_types` (16 MBTI codes & UNKNOWN)
  - may split into 4 seperat codes for E/I, S/N, T/F, J/P

Outcome for __'Outwit, Outplay, Outlast'__
- `order` (player longveity)
- `day` (days survived)
- `total_votes_received` (social risks)
- `
- `immunity_wins` (game perfomance, derived from `challenges_cleaned_df`)
- `reward_wins` (game perfomance, derived from `challenges_cleaned_df`)

Outcome
- `is_winner` (bool, derived from `castaway` == `winner`)

Time Axis for __'Evolution'__
- `season_year` (derived form `premiered`)

# Aggregate Castaway Dataframe

Create outplay_df to measure player's abbility to outplay others

From challenges_cleaned_df:
1. Group by `season`, `winners`, `challenge_type`, and count rows.
2. Pivot `challenge_type` to columns named `immunity_challange_wins` and `reward_challenge_wins`.
3. Reset the index and rename `winners` to `castaway`.

**Result**: a small lookup table with season, castaway, immunity wins, and reward wins.

In [ ]:
# Group by "season", "winners", "challenge_type" then count rows
outplay_df = challenges_cleaned_df.groupby(['season', 'winners', 'challenge_type']).size().reset_index(name='count')

# Pivot challenge_type to columns named immunity and reward
outplay_df = outplay_df.pivot(index=['season', 'winners'], columns='challenge_type', values='count').fillna(0).astype(int)

# Reset the index and rename winners to castaway
outplay_df = outplay_df.reset_index().rename(columns={'winners': 'castaway', 'immunity': 'immunity_challenge_wins', 'reward': 'reward_challenge_wins'})

# Remove challenge_type row label
outplay_df.columns.name = None

# Show Transformed Data
outplay_df.info()
outplay_df.head()


# Filter the Summary Dataframe

Create identity_df to for winner identification over time

From clean_summary_df:
1. Copy only `season`, `winner`, `premiered`.
2. Derive `season_year` from `premiered.dt.year` and drop `premiered`.

Result: identity_df with `season`, `winner`, `season_year`.

In [ ]:
# Copy only "season", "winner", "premiered"
identity_df = clean_summary_df[["season", "winner", "premiered"]].copy()
# Derive season_year from premiered.dt.year and drop premiered.
identity_df["season_year"] = identity_df["premiered"].dt.year
identity_df = identity_df.drop(columns=["premiered"])

# Show Transformed Data
identity_df.info()
identity_df.head()

# Filter Castaway Rows 

Create a traits_df for outlast, outwit, and personality analysis

From clean_castaways_df:
1. keep only: `season`, `castaway`, `full_name`, `personality_type`, `order`, `day`, `total_votes_received`, `immunity_idols_won`.
2. Rename `immunity_idols_won` → `immunity_idols_obtained`.

In [ ]:
# Filter the Castaway Dataframe
traits_df = clean_castaways_df[["season", "castaway", "full_name", "personality_type", "order", "day", "total_votes_received", "immunity_idols_won"]].copy()
traits_df.rename(columns={"immunity_idols_won": "immunity_idols_obtained"}, inplace=True)

# Show Transformed Data
traits_df.info()
traits_df.head()

# Merge New Dataframes for Analysis

1. Left-merge identity_df on season.
2. Left-merge the outplay_df on `season`, `castaway`.
3. fillna(0) on immunity_challenge_wins and reward_challenge_wins, cast to int for no win scenarios
4. Normalize `castaway` and `winner`
5. Create is_winner = normalized_castaway == normalized_winner.
6. Drop winner after the boolean is set.

In [ ]:
# left-merge identity_df on season
analysis_df = pd.merge(traits_df, identity_df, on='season', how='left')

# left-merge outplay_df on `season`, `castaway`
analysis_df = pd.merge(analysis_df, outplay_df, on=['season', 'castaway'], how='left')

# Fill NA values with 0 for immunity_challenge_wins and reward_challenge_wins
analysis_df[['immunity_challenge_wins', 'reward_challenge_wins']] = analysis_df[['immunity_challenge_wins', 'reward_challenge_wins']].fillna(0).astype(int)

# Normalize `castaway` and `winner`
analysis_df['castaway'] = analysis_df['castaway'].str.strip().str.casefold()
analysis_df['winner'] = analysis_df['winner'].str.strip().str.casefold()

# Create is_winner = normalized_castaway == normalized_winner.
analysis_df['is_winner'] = analysis_df['castaway'] == analysis_df['winner']

# Drop winner and castaway short names after the boolean is set.
analysis_df = analysis_df.drop(columns=['winner', 'castaway'])

# Re-order columns to something more intuitive
analysis_df = analysis_df[['season', 'full_name', 'personality_type', 'order', 'day', 'total_votes_received', 'immunity_idols_obtained', 'immunity_challenge_wins', 'reward_challenge_wins', 'is_winner', 'season_year']]

### Re-entry Season Mechanic

During season 38, one player was removed, re-entered and then won. The following cell attempts to remove duplicate winner entries, for all such cases.

In [ ]:
# Select duplicate winners
winner_mask = analysis_df["is_winner"]
dup_season_mask = analysis_df.groupby("season")["is_winner"].transform("sum") > 1
dup_mask = winner_mask & dup_season_mask

# Select index of duplicate winners
dup_index = analysis_df.loc[dup_mask].index

# Select index ofthe highest order winner
keep_index = analysis_df.loc[dup_mask].groupby(["season", "full_name"])["order"].idxmax()

# Drop duplicate winner rows that are not the highest order winner
drop_index = dup_index.difference(keep_index.values)
analysis_df = analysis_df.drop(drop_index)

# Show final anlaysis dataframe
analysis_df.info()
analysis_df.head()

# Initial Statistics
- Determine the  global mean, median, and standard deviations for key metrics
- Compute these again for winners only
- Compute them again for era
- Focused correlations for gameplay, logevity, and is_winner

## Global Statiscs - Everyone vs. Winners

In [ ]:
# Key columns for statistics
stats_cols = ["order", "day", "total_votes_received", "immunity_idols_obtained", "immunity_challenge_wins", "reward_challenge_wins"]

# Global Statistics for Analysis_df
print("Global Statistics for Analysis_df")
display(HTML(analysis_df[stats_cols].describe().style.format(precision=2).to_html()))

# Winners Only Statistics
print("\nWinners Only Statistics")
winner_df = analysis_df.loc[analysis_df["is_winner"], stats_cols]
display(HTML(winner_df.describe().style.format(precision=2).to_html()))

### Statistics Initial Thoughts

Everyone
- Order is essentially uniform
- Day clusters at the beginning and in finals ranges
- Votes received is skewed to the right, so a few people must collect many votes
- Idols obtained has an inflation of zeros, due to unavailbility in early seasons

Winners
- Go twice as far (day and order)
- Recieve about half the number of votes
- Roughly double the idols and challenge wins
- Winning requires surviving, so any metric that grows with longevity should be normalized


## Correlations

In [ ]:
# Correlation columns
stats_cols = ["order", "day", "total_votes_received", "immunity_idols_obtained", "immunity_challenge_wins", "reward_challenge_wins", "is_winner"]
print("Correlation Matrix for Everyone")
display(HTML(analysis_df[stats_cols].corr().style.format(precision=2).to_html()))

### Correlation Thoughts
Order and Day 
- Are essentially redundant with a 0.96 for showing longevity

Votes received 
- Is uncorrelated with all almost all statistics
- Has a modest negative correlation with winners
- Winners most likely avoid getting as many votes

Idols Obtained 
- Is the strongest indicator 
- Moderate correlation to longevity (order/days) 
- Modestly with winners 
- None with votes received
- They may matter more for longevity than for social risks.

Immunity and Reward Wins
- Track strongly with day, most likely due to tenure
- Only modestly for winners
- They appear to keep you in the game longer, but don't necessarily lead to winning.

Winners
- Modestly recieve less votes and more challenge wins
- Moderately receive idols

# Visual 1: Winners vs. Non-Winners across Eras
- How do winners and non-winners compare on average idols, immunity wins, and challenge wins across different eras?

## Strategy
1. Drop seasons with re-entry
2. Create season bins for each era (every 5 years)
3. Group by era and winner
4. Show key metrics

In [ ]:
outplay_analysis_df = analysis_df.copy()

#drop seasons with re-entry abilities, skews the data
outplay_analysis_df =outplay_analysis_df[~outplay_analysis_df["season"].isin([22, 23, 27, 38, 40])]

#sort from low to high seasons
outplay_analysis_df = outplay_analysis_df.sort_values(by="season")

#create bins to group seasons into eras of the game
season_bins = [1999, 2004, 2009, 2014, 2020]
season_labels = ["2000-2004", "2005-2009", "2010-2014", "2015-2020"]

#add new column for eras
outplay_analysis_df["era"] = pd.cut(
    outplay_analysis_df["season_year"],
    bins = season_bins,
    labels = season_labels
)

era_means = outplay_analysis_df.groupby(["era", "is_winner"])[["immunity_challenge_wins", "reward_challenge_wins", "immunity_idols_obtained"]].mean()
era_means = era_means.reset_index()

#manually edit immunity_idols for 2000-2004, there were no idols introduced until 2005
era_means.loc[era_means["era"] == "2000-2004", "immunity_idols_obtained"] = 0

#visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.barplot(
    data=era_means,
    x="era",
    y="immunity_challenge_wins",
    hue="is_winner",
    ax=axes[0]
)

sns.barplot(
    data=era_means,
    x="era",
    y="reward_challenge_wins",
    hue="is_winner",
    ax=axes[1]
)

sns.barplot(
    data=era_means,
    x="era",
    y="immunity_idols_obtained",
    hue="is_winner",
    ax=axes[2]
)


#formatting
axes[0].set_title("Immunity Challenge Wins by Era")
axes[0].set_xlabel("Era")
axes[0].set_ylabel("Mean Wins")

axes[1].set_title("Reward Challenge Wins by Era")
axes[1].set_xlabel("Era")
axes[1].set_ylabel("Mean Wins")

axes[2].set_title("Immunity Idols Obtained by Era")
axes[2].set_xlabel("Era")
axes[2].set_ylabel("Mean Idols")

#normalize challenge and reward wins to the same y range
axes[0].set_ylim(0, 8)
axes[1].set_ylim(0, 8)
axes[2].set_ylim(0, 2.5)

#remove redundant labels
axes[0].get_legend().remove()
axes[1].get_legend().remove()
axes[2].legend(title="Winner", labels=["Non-Winner", "Winner"], handles=axes[2].legend_.legend_handles)

#final formatting
fig.suptitle("Winner vs Non-Winner Performance Across Eras", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


*Footnote: Immunity idols were not introduced into survivor unti Season 11 in 2005, but the dataframe has values for idols in the 2000-2004 era. This may be representing other advantages, but for our purposes I manually zero-ed out this era to avoid confusion.*

## Observations

The purpose of this visualization was to compare what winners versus losers of Survivor looked like over the different eras of survivor. We were expecting to see challenge and reward wins to decrease over the eras as strategy became more complicated and intense, and players seemed more focused on eliminating strong competitors earlier. 

Surprisingly, based on the graphs we can see that winners have dominated immunity challenges around 2x more, and reward challenges around 2.3x more. This directly challenges the expectation that "challenge-beasts" don't win Survivor. The only exception is the pre-idol era (2000-2004), where the gap between losers and winners is smaller. 

When idols were introduced, it seems like winners had the protection of idols to play more dominantly in challenges without the fear of being voted off. Additonally, players who had both idols and were winning immunity got harder and harder to vote out. This not only proves the outplay component, but also the outlast and outwit. Based on the graph, winners find around 3x more idols than losers. To find idols you have to outwit opponents by understanding where clues may be hidden, and then figuring out how to go look for an idol without your teammates catching on. Additionally, the added protection of an idol allows a player to outlast even when being voted off. 

It should be acknowleged that the winner sample is much smaller per era (8-10 winners versus 125-171 losers), which could skew data. Within the losers sample are contestants who only lasted a few days versus contestants who lasted until the end. 

# Visual 2: Personality Type (MBTI) & Winners
- How dow the MBTI codes correlate with winners?

# Strategy
1. Drop unkowns
2. Split personality MBTI code into 4 boolean letter codes
3. Normalize columns that naturally grow with time

In [ ]:
# Create a copy for the MBTI correlation heatmap
mbti_heatmap_df = analysis_df.copy()

# Rename idol column if the notebook used a different final name
if "immunity_idols_obtained" not in mbti_heatmap_df.columns:
    if "immunity_idols_found" in mbti_heatmap_df.columns:
        mbti_heatmap_df = mbti_heatmap_df.rename(
            columns={"immunity_idols_found": "immunity_idols_obtained"}
        )

# Keep only rows with a real MBTI type
mbti_heatmap_df = mbti_heatmap_df[
    mbti_heatmap_df["personality_type"] != "UNKNOWN"
].copy()

# Create MBTI letter flags
# These use one side of each MBTI pair to avoid duplicate opposite columns
mbti_heatmap_df["is_extrovert"] = (
    mbti_heatmap_df["personality_type"].str[0] == "E"
).astype(int)

mbti_heatmap_df["is_intuitive"] = (
    mbti_heatmap_df["personality_type"].str[1] == "N"
).astype(int)

mbti_heatmap_df["is_thinking"] = (
    mbti_heatmap_df["personality_type"].str[2] == "T"
).astype(int)

mbti_heatmap_df["is_judging"] = (
    mbti_heatmap_df["personality_type"].str[3] == "J"
).astype(int)

# Convert winner label to 0/1 for correlation
mbti_heatmap_df["winner_flag"] = mbti_heatmap_df["is_winner"].astype(int)

# Normalize metrics that naturally grow with time in the game
mbti_heatmap_df["survival_pct"] = (
    mbti_heatmap_df["order"] /
    mbti_heatmap_df.groupby("season")["order"].transform("max")
)

mbti_heatmap_df["votes_per_day"] = (
    mbti_heatmap_df["total_votes_received"] /
    mbti_heatmap_df["day"].clip(lower=1)
)

mbti_heatmap_df["idols_per_day"] = (
    mbti_heatmap_df["immunity_idols_obtained"] /
    mbti_heatmap_df["day"].clip(lower=1)
)

mbti_heatmap_df["immunity_wins_per_day"] = (
    mbti_heatmap_df["immunity_challenge_wins"] /
    mbti_heatmap_df["day"].clip(lower=1)
)

mbti_heatmap_df["reward_wins_per_day"] = (
    mbti_heatmap_df["reward_challenge_wins"] /
    mbti_heatmap_df["day"].clip(lower=1)
)

# Select MBTI dimensions and outcome/gameplay metrics
mbti_columns = [
    "is_extrovert",
    "is_intuitive",
    "is_thinking",
    "is_judging"
]

metric_columns = [
    "winner_flag",
    "survival_pct",
    "votes_per_day",
    "idols_per_day",
    "immunity_wins_per_day",
    "reward_wins_per_day"
]

# Create a focused correlation table
mbti_correlation_matrix = mbti_heatmap_df[
    mbti_columns + metric_columns
].corr()

mbti_correlation_focus = mbti_correlation_matrix.loc[
    mbti_columns,
    metric_columns
].round(2)

mbti_correlation_focus

In [ ]:
mbti_correlation_plot = mbti_correlation_focus.rename(
    index={
        "is_extrovert": f"Extrovert (+) vs.\nIntrovert (-)",
        "is_intuitive": f"Intuitive (+) vs.\nSensing (-)",
        "is_thinking": f"Thinking (+) vs.\nFeeling (-)",
        "is_judging": f"Judging (+) vs.\nPerceiving (-)"
    },
    columns={
        "winner_flag": "Winner",
        "survival_pct": "Survival %",
        "votes_per_day": "Votes per day",
        "idols_per_day": "Idols per day",
        "immunity_wins_per_day": "Immunity wins per day",
        "reward_wins_per_day": "Reward wins per day"
    }
)

# Draw Heat Map
fig, ax = plt.subplots(figsize=(12, 5.5))

sns.heatmap(
    data=mbti_correlation_plot,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    center=0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Curtailed Correlation"}
)

ax.set_title(
    "MBTI Letter Correlation With Survivor Performance Metrics",
    fontsize=16,
    pad=14
)

ax.set_xlabel("Performance Metrics", labelpad=12)
ax.set_ylabel("MBTI Letter Flags", labelpad=18)

ax.tick_params(axis="x", labelrotation=35, labelsize=10)
ax.tick_params(axis="y", labelrotation=0, labelsize=11, pad=8)

plt.tight_layout()
plt.show()

### MBTI Correlation Heatmap Observation

This heatmap compares MBTI letter dimensions against winner status and normalized Survivor performance metrics. I split each personality type into broader letter flags because the full four-letter MBTI groups are small, while the Extrovert/Introvert (E/I), Sensing/Intuitive (S/N), Thinking/Feeling (T/F), and Judging/Perceiving (J/P) dimensions give a cleaner view of the data.

The correlations are mostly weak, which means MBTI alone does not strongly explain who wins or performs well. Still, a few small patterns are worth noticing. Thinking types show the strongest positive relationship with winner status and idols per day, while judging types trend slightly lower across challenge-win metrics. These are not definitive conclusions, but they give us a useful starting point for asking better questions.

I also normalized votes, idols, and challenge wins by days survived because raw totals naturally favor players who lasted longer. That keeps the visual more objectively correct. Instead of treating personality as the be all and end all, this heatmap treats it as one measure among many being that of a small signal inside the much bigger social game.

In [ ]:
print("MBTI heatmap rows:", len(mbti_heatmap_df))
print("\nColumns used:")
print(mbti_columns + metric_columns)

print("\nMissing values:")
print(mbti_heatmap_df[mbti_columns + metric_columns].isna().sum())

# Visual 3: Winners & Votes
- How are winners receiving votes compared to non-winners?

## Strategy
1. Normalize total votes received according to time
2. Use a violin chart to illustrate

In [ ]:
#Create a copy of the analysis dataframe
votes_outcome_df = analysis_df.copy()

# Normalize votes by days survived
votes_outcome_df["votes_per_day"] = (
    votes_outcome_df["total_votes_received"] /
    votes_outcome_df["day"].clip(lower=1)
)

vote_cap = votes_outcome_df["votes_per_day"].quantile(0.99)
votes_outcome_df["vote_plot"] = votes_outcome_df["votes_per_day"].clip(upper=vote_cap)

# Violin plot
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(
    x="is_winner",
    y="vote_plot",
    ax=ax,
    inner="box",
    data=votes_outcome_df,
)

# Formatting
ax.set_title("Votes per Day Survived by Winner Status")
ax.set_xlabel("Winner")
ax.set_ylabel("Votes per Day")
fig.text(0.5, -0.1, "Note: Capping at 99th percentile to reduce outlier influence", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## Observation

### Notes
From the early global statistics, the non-winners have a approximately double the median and mean votes received than winners. The violin chart above accounts for votes received per day, to focus in on the rate of accumulation rather than total votes received. 

Limitations: The non-winner group mixes in people who were voted off early, so the votes recieved per day has big swings, and a few people collect many of votes. The length of its violin partly reflects game length and how often someone is targeted.

### Thoughts
It appears that the majority of winners receive a very low number of votes per day, whereas non-winners have a wider distribution of votes at a higher rate per day then winners.

# Simeon's Question
**Fill Below**

# Question 1
How does the depth of game (calendar time, `days` or `order`) change the way we think about winners and everyone else?

### Motivation
It seems that time affects the comparisons between competitors more than any other feature. In order to combat this, the data should be normalized against time. Also, seasons should be binnned per era, and individual seasons should be binned by depth. This helps deal with changes in the game over eras, and accumulation of data during the length of seasons. For example, players in the first few seasons did not have access to idols, which is a key feature in later seasons. Also during the course of a season, players who are voted off early had no chance to accumulate any data on challenge wins.

### Step 1: Normalize Columns by Day and Order

- Normalizing by `day` will be used on vote, idols, and challenge wins.
- Normalizing by `order` will be used to determine survival depth and quartiles.


In [ ]:
# Create a copy of the analysis dataframe
depth_df = analysis_df.copy()

# Normalize columns that accumulate over time
num_days = depth_df["day"].clip(lower=1) # clip to avoid division by 0

depth_df["votes_per_day"] = depth_df["total_votes_received"] / num_days
depth_df["idols_per_day"] = depth_df["immunity_idols_obtained"] / num_days
depth_df["immunity_wins_per_day"] = depth_df["immunity_challenge_wins"] / num_days
depth_df["reward_wins_per_day"] = depth_df["reward_challenge_wins"] / num_days

# Calculate survival depth as a percentage of remaining players
# i.e. what percentage of players did they outlast?
# Higher order number means they lasted longer
max_players = depth_df.groupby("season")["order"].transform("max")
depth_df["survival_pct"] = depth_df["order"] / max_players

# Drop unneeded columns
depth_df = depth_df.drop(columns=["total_votes_received", "immunity_idols_obtained", "immunity_challenge_wins", "reward_challenge_wins"])

### Step 2: Bin seasons by Era

- Ideally this will be used for filtering out changes across seasons
  - Particularly the Pre-2005 era, which did not have access to idols
- Note: this transfomation matches the bin sizes used earlier

In [ ]:
# Create bins for each era (every 5 seasons)
era_bins = [1999, 2004, 2009, 2014, 2019]
era_labels = ["2000-2004", "2005-2009", "2010-2014", "2015-2020"]

# Bin the data
depth_df["era"] = pd.cut(depth_df["season_year"], bins=era_bins, labels=era_labels, right=False)

### Step 3: Bin Depth within each Season
- Early contestants have little to no collection of data when compared to contestants that lasted longer.
- Binning by depth allows for comparisons between players who survived a similar amount of time
- This allows for comparisons between players who were booted early, or for more specific analysis on players who made it almost to the end.

# Emma's Question
**Fill Below**

# Jarren's Question
**Fill Below**

## AI disclaimer

**Tool:** Cursor  

**Used for:** syntax questions, grammar, review, structure

**Scope:** refinement